# VNLegal RAG — Full Pipeline Runner (`src`)

Chạy **toàn bộ pipeline truy hồi** bằng cách gọi các hàm trong package `src/`, hiển thị output ở **từng bước**.

Artifacts mặc định lấy từ thư mục `model/`:
- TextCNN: `model/textcnn_artifacts` (8 nhãn macro_domain, `max_len=128`)
- Siamese: `model/siamese_lstm_traditional_cosine_artifacts` (LSTM cosine)
- Data: `data/data_ready`

4 mode truy hồi: `tfidf_textcnn`, `siamese_textcnn`, `siamese_only`, `hybrid_textcnn` (weighted-sum sparse+dense).

> Nếu CUDA local lỗi, ô dưới đặt `VNLEGAL_DEVICE=cpu` trước khi import `src`.

## 1. Setup — đưa repo root vào `sys.path`

In [1]:
import os, sys
from pathlib import Path

# Ép CPU nếu CUDA local hỏng; bỏ dòng này để tự phát hiện GPU.
os.environ.setdefault('VNLEGAL_DEVICE', 'cpu')

# Notebook nằm trong src/ → đi ngược lên tới repo root (thư mục chứa src/__init__.py).
repo_root = Path.cwd()
while not (repo_root / 'src' / '__init__.py').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo_root:', repo_root)

repo_root: C:\Users\hung\Documents\GitHub\vnlegal-rag


## 2. Build pipeline

Resolve data + artifacts, load TextCNN, dựng TF-IDF index, và (vì có Siamese weights) precompute document embeddings. Log từng giai đoạn in ra ngay dưới đây.

In [2]:
from src import PipelineConfig, RetrievalPipeline, build_markdown_table

config = PipelineConfig()
print('device:', config.device, '| hybrid_alpha:', config.hybrid_alpha)

pipe = RetrievalPipeline.from_config(config)
print()
print('TextCNN artifact :', pipe.textcnn['artifact_dir'])
print('Siamese artifact :', pipe.siamese['artifact_dir'] if pipe.siamese else None)
print('Siamese available:', pipe.has_siamese)
print('Corpus size      :', len(pipe.corpus_df))
print('Macro-domains    :', pipe.label_list)

device: cpu | hybrid_alpha: 0.7


[data] dir=data_ready  corpus=9715  qa_test=2991  labels=8
[textcnn] textcnn_artifacts  max_len=128


[tfidf] matrix=(9715, 6033)


[siamese] loaded siamese_lstm_traditional_cosine_artifacts

TextCNN artifact : C:\Users\hung\Documents\GitHub\vnlegal-rag\model\textcnn_artifacts
Siamese artifact : C:\Users\hung\Documents\GitHub\vnlegal-rag\model\siamese_lstm_traditional_cosine_artifacts
Siamese available: True
Corpus size      : 9715
Macro-domains    : ['Civil & Investment', 'Finance & Banking', 'Industry, Resources & Environment', 'Justice & Dispute Resolution', 'Labor & Insurance', 'Security & Defense', 'State Organization & Admin', 'Transportation']


## 3. Dự đoán chủ đề (TextCNN topic filter)

In [3]:
from src.retrieval import predict_topic_topk

question = 'Hợp đồng lao động thời vụ có bắt buộc đóng bảo hiểm xã hội không?'
ids, labels, probs = predict_topic_topk(question, pipe.textcnn, pipe.config.device, k=3)
print('Câu hỏi:', question)
print('Top-3 chủ đề dự đoán:')
for lab, p in zip(labels, probs):
    print(f'  {lab:35s} {float(p):.3f}')

Câu hỏi: Hợp đồng lao động thời vụ có bắt buộc đóng bảo hiểm xã hội không?
Top-3 chủ đề dự đoán:
  Labor & Insurance                   0.972
  Civil & Investment                  0.006
  Security & Defense                  0.006


## 4. Truy hồi: TF-IDF + TextCNN (top-5)

In [4]:
display(pipe.retrieve_tfidf_textcnn(question, k=5))

,score,passage_id,macro_domain,article_content
0,0.790364,du_thao_luat_bao_hiem_xa_hoi_dieu_36_71b61966,Labor & Insurance,"Điều 36. Chậm đóng, trốn đóng bảo hiểm xã hội ..."
1,0.704657,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 13. Trách nhiệm của người sử dụng lao độn...
2,0.689870,du_thao_luat_bao_hiem_xa_hoi_dieu_110_d869b78c,Labor & Insurance,Điều 110. Chế độ hưu trí và chế độ tử tuất đối...
3,0.679229,du_thao_luat_bao_hiem_xa_hoi_dieu_12_6228f499,Labor & Insurance,Điều 12. Trách nhiệm của người sử dụng lao độn...
4,0.669796,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 111. Chế độ hưu trí và chế độ tử tuất đối...


## 5. Truy hồi: Siamese + TextCNN (top-5)

In [5]:
display(pipe.retrieve_siamese_textcnn(question, k=5))

,score,passage_id,macro_domain,article_content
0,0.547702,du_thao_luat_bao_hiem_xa_hoi_dieu_36_71b61966,Labor & Insurance,"Điều 36. Chậm đóng, trốn đóng bảo hiểm xã hội ..."
1,0.441822,luat_cong_nghiep_quoc_phong_an_ninh_va_ong_vie...,Security & Defense,Điều 62. Chính sách đối với cơ sở công nghiệp ...
2,0.424683,du_thao_luat_kinh_doanh_bao_hiem_sua_oi_dieu_1...,Labor & Insurance,Điều 153. Tách nguồn vốn chủ sở hữu và nguồn p...
3,0.417430,luat_dan_quan_tu_ve_cua_quoc_hoi_so_43_2009_qh...,Security & Defense,Điều 52. Nguồn kinh phí\r\n\r\n1. Kinh phí cho...
4,0.414221,luat_can_cuoc_cong_dan_cua_quoc_hoi_so_59_2014...,Civil & Investment,"Điều 28. Thu hồi, tạm giữ thẻ Căn cước công dâ..."


## 6. Truy hồi: Siamese only (top-5)

In [6]:
display(pipe.retrieve_siamese_only(question, k=5))

,score,passage_id,macro_domain,article_content
0,0.547702,du_thao_luat_bao_hiem_xa_hoi_dieu_36_71b61966,Labor & Insurance,"Điều 36. Chậm đóng, trốn đóng bảo hiểm xã hội ..."
1,0.535900,luat_au_gia_tai_san_cua_quoc_hoi_so_01_2016_qh...,Finance & Banking,"Điều 66. Thù lao dịch vụ đấu giá, chi phí đấu ..."
2,0.501198,luat_au_gia_tai_san_cua_quoc_hoi_so_01_2016_qh...,Finance & Banking,Điều 24. Quyền và nghĩa vụ của tổ chức đấu giá...
3,0.475777,luat_au_gia_tai_san_cua_quoc_hoi_so_01_2016_qh...,Finance & Banking,"Điều 68. Quản lý, sử dụng thù lao dịch vụ đấu ..."
4,0.475431,luat_cong_chung_cua_quoc_hoi_so_53_2014_qh13_d...,Justice & Dispute Resolution,Điều 32. Quyền của tổ chức hành nghề công chứn...


## 7. Truy hồi: Hybrid (TF-IDF + Siamese, weighted-sum) (top-5)

Trên cùng pool topic-filter: `score = α·norm(tfidf) + (1−α)·norm(siamese)`, `α = config.hybrid_alpha` (mặc định 0.7, nghiêng về TF-IDF).

In [7]:
print('alpha =', pipe.config.hybrid_alpha)
display(pipe.retrieve_hybrid_textcnn(question, k=5))

alpha = 0.7


,score,passage_id,macro_domain,article_content
0,1.000000,du_thao_luat_bao_hiem_xa_hoi_dieu_36_71b61966,Labor & Insurance,"Điều 36. Chậm đóng, trốn đóng bảo hiểm xã hội ..."
1,0.879162,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 13. Trách nhiệm của người sử dụng lao độn...
2,0.825169,du_thao_luat_bao_hiem_xa_hoi_dieu_110_d869b78c,Labor & Insurance,Điều 110. Chế độ hưu trí và chế độ tử tuất đối...
3,0.816299,du_thao_luat_bao_hiem_xa_hoi_dieu_12_6228f499,Labor & Insurance,Điều 12. Trách nhiệm của người sử dụng lao độn...
4,0.806879,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 111. Chế độ hưu trí và chế độ tử tuất đối...


## 8. Đánh giá trên `qa_test` — 4 mode

Recall@{1,3,5,10} và MRR trên mẫu con. `hybrid_textcnn` xuất hiện cạnh 3 mode gốc.

In [8]:
from IPython.display import Markdown, display

pipe.config.eval_sample_n = 200   # đặt None để chạy toàn bộ split
results = pipe.evaluate('test')
display(Markdown(build_markdown_table(results)))

| pipeline        |   n_queries |   Recall@1 |   Recall@3 |   Recall@5 |   Recall@10 |    MRR |
|:----------------|------------:|-----------:|-----------:|-----------:|------------:|-------:|
| tfidf_textcnn   |         200 |     0.4800 |     0.6500 |     0.7250 |      0.7800 | 0.5779 |
| siamese_textcnn |         200 |     0.3000 |     0.4600 |     0.5450 |      0.6350 | 0.4044 |
| siamese_only    |         200 |     0.3350 |     0.4850 |     0.5650 |      0.6250 | 0.4277 |
| hybrid_textcnn  |         200 |     0.5150 |     0.6900 |     0.7550 |      0.8200 | 0.6172 |